# BioOperatorEnv — Before/After Demo

Runs the **untrained** Qwen baseline and the **trained** (LoRA) Qwen on the SAME seeded scenario, side-by-side. Generates the headline plot for the README/video.

Requires the LoRA adapter from `notebooks/03_train_grpo.ipynb` (or any compatible adapter under `checkpoints/qwen3-bioperator-lora`).

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q transformers peft torch accelerate bitsandbytes
!pip install -q numpy scipy pandas pydantic matplotlib tqdm fastapi uvicorn

In [ ]:
import os
REPO_URL = 'https://github.com/Json604/openenv-bioreactor.git'
REPO_DIR = 'openenv-bioreactor'
if not os.path.exists(REPO_DIR) and not os.path.exists('bioperator_env'):
    !git clone $REPO_URL $REPO_DIR

In [ ]:
import os, sys
from pathlib import Path
candidates = [Path.cwd(), Path.cwd() / 'openenv-bioreactor', Path.cwd() / 'meta_env', Path.cwd().parent]
repo_root = next((p for p in candidates if (p / 'bioperator_env').exists()), None)
assert repo_root is not None, f'cannot locate bioperator_env (cwd={Path.cwd()})'
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
print('repo root:', repo_root)

from baselines.untrained_qwen_agent import UntrainedQwenAgent
from baselines.trained_qwen_agent import TrainedQwenAgent
from bioperator_env.env import BioOperatorEnv
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
TASK = 'do-recovery-medium'
SEED = 999

def run_episode(agent_cls, **kw):
    env = BioOperatorEnv(task_id=TASK, seed=SEED)
    obs = env.reset()
    agent = agent_cls(**kw)
    do, feed, aer, rewards, sample_actions = [], [], [], [], []
    done = False
    while not done:
        a = agent.act(obs)
        obs, r, done, info = env.step(a)
        do.append(obs.measurements['dissolved_oxygen_pct'])
        feed.append(a['feed_delta_L_h'])
        aer.append(a['aeration_delta_vvm'])
        rewards.append(r)
        if env.step_count <= 3:
            sample_actions.append((env.step_count, a))
    return dict(do=do, feed=feed, aer=aer, rewards=rewards,
                cum=sum(rewards), sample_actions=sample_actions,
                violations=env.safety_violations)

untrained = run_episode(UntrainedQwenAgent)
trained = run_episode(TrainedQwenAgent)
print('untrained: cum=', untrained['cum'], 'violations=', untrained['violations'])
print('trained:   cum=', trained['cum'], 'violations=', trained['violations'])

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(untrained['do'], label='untrained Qwen', linewidth=2, color='#e34a33')
axes[0].plot(trained['do'], label='trained Qwen (GRPO+LoRA)', linewidth=2, color='#31a354')
axes[0].axhline(20, color='grey', linestyle='--', label='DO_min_safe=20%')
axes[0].set_ylabel('Dissolved O2 %')
axes[0].set_title(f'Before/After: {TASK} (seed={SEED})')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(np.cumsum(untrained['rewards']), label='untrained cum reward', color='#e34a33')
axes[1].plot(np.cumsum(trained['rewards']), label='trained cum reward', color='#31a354')
axes[1].set_xlabel('Episode step')
axes[1].set_ylabel('Cumulative reward')
axes[1].legend(); axes[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig('results/before_after_demo.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
import json
examples = ['# Before / After Action Examples\n']
for label, run in [('Untrained Qwen', untrained), ('Trained Qwen', trained)]:
    examples.append(f'\n## {label}\n')
    for step, a in run['sample_actions']:
        examples.append(f'\n**Step {step}:** ```json\n{json.dumps(a, indent=2)}\n```')
with open('results/before_after_action_examples.md', 'w') as f:
    f.writelines(examples)
print('wrote results/before_after_action_examples.md')